# Aula 18 — Deterministic Workflows

Na Aula 17, aprendemos a expor capacidades como tools.

Agora vamos combinar essas capacidades em uma sequência explícita:

```text
tool
→ tool
→ regra
→ estado
→ checkpoint
→ decisão
→ próxima etapa
```

A pergunta central é:

> **Como coordenar múltiplas capacidades de forma confiável sem precisar de um agente autônomo?**


## Objetivos

Ao final da aula, você deverá conseguir:

- distinguir tool de workflow;
- explicar o que torna um workflow determinístico;
- representar estado explicitamente;
- implementar retry com limite;
- explicar idempotência;
- registrar checkpoints;
- aplicar approval gates;
- retomar um fluxo após falha;
- observar a execução por etapa;
- justificar quando um workflow é suficiente e um agente ainda não é necessário.


## Glossário da aula

Conceitos centrais no **Glossário Vivo**:

**[Contrato de decisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#contrato-de-decisão) · [Decisão tipada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#decisão-tipada) · [Roteamento por confiança](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-por-confiança) · [Política de roteamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#política-de-roteamento) · [Workflow determinístico](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#workflow-determinístico) · [Estado do workflow](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#estado-do-workflow) · [Retry](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#retry) · [Idempotência](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#idempotência) · [Checkpoint](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#checkpoint) · [Erro recuperável](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#erro-recuperável) · [Observabilidade de workflows](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#observabilidade-de-workflows) · [Gate de aprovação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#gate-de-aprovação)**

> Use os links ao longo da aula para revisar a definição formal no momento em que cada conceito aparece.


## 1. Tool vs workflow

Uma **tool** representa uma capacidade.

Um **workflow** coordena capacidades sob uma sequência definida.

```text
tool
→ faz uma coisa

workflow
→ coordena várias coisas
```

Exemplo:

```text
get_lesson_status
→ build_release_note
→ approval_gate
→ publish_release_note
```

Na Aula 18, a ordem é conhecida previamente. O sistema não improvisa o próximo passo.


In [ ]:
from dataclasses import dataclass, field
from time import perf_counter
from typing import Any, Callable
import copy
import pandas as pd

print("Ambiente da Aula 18 carregado.")


## 2. [Estado do workflow](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#estado-do-workflow)

Para executar um fluxo de forma confiável, precisamos saber **onde estamos**.

Vamos carregar um estado explícito com:

```text
workflow_id
current_step
status
inputs
outputs
attempts
errors
approved
checkpoints
```

O estado evita que a execução dependa de variáveis implícitas espalhadas pelo notebook.


In [ ]:
@dataclass
class WorkflowState:
    workflow_id: str
    lesson_id: str
    approved: bool | None = None
    current_step: str = "start"
    status: str = "pending"
    outputs: dict = field(default_factory=dict)
    attempts: dict = field(default_factory=dict)
    errors: list = field(default_factory=list)
    checkpoints: list = field(default_factory=list)
    events: list = field(default_factory=list)


## 3. Tools locais do laboratório

O laboratório usa um fluxo didático de liberação de aula:

```text
validate_release_request
→ get_lesson_status
→ build_release_note
→ approval_gate
→ publish_release_note
```

A publicação final é simulada em memória. Nenhum efeito externo real será produzido.


In [ ]:
LESSON_STATUS = {
    "16": "Available",
    "17": "Available",
    "18": "Draft",
}

PUBLISHED_NOTES = {}

def validate_release_request(state: WorkflowState):
    if state.lesson_id not in LESSON_STATUS:
        raise ValueError("lesson_not_found")
    return {"validated": True}

def get_lesson_status(state: WorkflowState):
    return {"lesson_status": LESSON_STATUS[state.lesson_id]}

def build_release_note(state: WorkflowState):
    status = state.outputs["get_lesson_status"]["lesson_status"]
    return {"release_note": f"Aula {state.lesson_id}: status atual = {status}."}

def approval_gate(state: WorkflowState):
    if state.approved is True:
        return {"approval": "approved"}
    if state.approved is False:
        raise PermissionError("approval_rejected")
    raise PermissionError("approval_required")

def publish_release_note(state: WorkflowState):
    note = state.outputs["build_release_note"]["release_note"]
    if state.workflow_id in PUBLISHED_NOTES:
        return {"publication": "already_published", "note": PUBLISHED_NOTES[state.workflow_id]}
    PUBLISHED_NOTES[state.workflow_id] = note
    return {"publication": "published", "note": note}


## 4. Workflow runner

O runner será responsável por:

- atualizar estado;
- contar tentativas;
- registrar eventos;
- capturar erros;
- criar checkpoints;
- interromper o fluxo quando necessário.

Essa separação é importante:

```text
regra do workflow
≠
lógica da tool
```


In [ ]:
WORKFLOW_STEPS = [
    ("validate_release_request", validate_release_request),
    ("get_lesson_status", get_lesson_status),
    ("build_release_note", build_release_note),
    ("approval_gate", approval_gate),
    ("publish_release_note", publish_release_note),
]

def checkpoint(state: WorkflowState, label: str):
    state.checkpoints.append({
        "label": label,
        "current_step": state.current_step,
        "status": state.status,
        "outputs": copy.deepcopy(state.outputs),
        "attempts": copy.deepcopy(state.attempts),
    })

def run_step(state: WorkflowState, step_name: str, fn: Callable):
    state.current_step = step_name
    state.attempts[step_name] = state.attempts.get(step_name, 0) + 1
    attempt = state.attempts[step_name]
    start = perf_counter()

    event = {
        "workflow_id": state.workflow_id,
        "step_name": step_name,
        "attempt": attempt,
        "status": "running",
        "error_type": None,
        "latency_ms": None,
    }

    try:
        result = fn(state)
        state.outputs[step_name] = result
        event["status"] = "ok"
        state.status = "running"
        checkpoint(state, f"after:{step_name}")
        return True
    except Exception as exc:
        error_name = type(exc).__name__
        event["status"] = "failed"
        event["error_type"] = error_name
        state.errors.append({
            "step": step_name,
            "attempt": attempt,
            "error": repr(exc),
        })
        state.status = "paused"
        return False
    finally:
        event["latency_ms"] = (perf_counter() - start) * 1000
        state.events.append(event)

def run_workflow(state: WorkflowState, start_at: int = 0):
    state.status = "running"

    for index in range(start_at, len(WORKFLOW_STEPS)):
        name, fn = WORKFLOW_STEPS[index]
        ok = run_step(state, name, fn)
        if not ok:
            return state

    state.current_step = "complete"
    state.status = "complete"
    checkpoint(state, "workflow:complete")
    return state


## 5. Happy path

Primeiro vamos executar o fluxo com aprovação já concedida.

Observe:

- a ordem das etapas;
- os outputs acumulados;
- os checkpoints;
- o log de eventos.


In [ ]:
happy = WorkflowState(
    workflow_id="WF-001",
    lesson_id="17",
    approved=True,
)

run_workflow(happy)

print("status:", happy.status)
print("current_step:", happy.current_step)
display(pd.DataFrame(happy.events))


## 6. [Checkpoint](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#checkpoint)

Um checkpoint registra um estado intermediário.

Ele permite responder:

```text
até onde o workflow chegou?
o que já foi concluído?
quais outputs existem?
quantas tentativas ocorreram?
```

Isso é diferente de simplesmente repetir tudo desde o início.


In [ ]:
checkpoint_view = pd.DataFrame([
    {
        "label": item["label"],
        "current_step": item["current_step"],
        "status": item["status"],
        "known_outputs": list(item["outputs"].keys()),
    }
    for item in happy.checkpoints
])

display(checkpoint_view)


## 6A. [Decision Contract / Contrato de decisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#contrato-de-decisão)

Um workflow também pode receber decisões produzidas por um modelo, regra ou classificador.

Mas a decisão deve entrar no fluxo como **dado estruturado**, não como instrução implícita.

Vamos usar:

```text
choice
confidence
source
latency_ms
cost
```

A ideia central é:

```text
decision generation
→ typed decision
→ routing policy
→ execution
```

Isso separa **quem produz a decisão** de **quem decide o que fazer com ela**.

> Nesta aula, ensinamos a abstração. Uma ferramenta como Jev poderia futuramente implementar o papel de `DecisionProvider`, mas o workflow não depende dela.


In [ ]:
@dataclass
class Decision:
    choice: str
    confidence: float
    source: str
    latency_ms: float
    cost: float

def validate_decision(decision: Decision):
    if not 0.0 <= decision.confidence <= 1.0:
        raise ValueError("confidence_out_of_range")
    if not decision.choice:
        raise ValueError("empty_choice")
    return True


## 6B. [Confidence-Gated Routing](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-por-confiança)

A confiança não executa nada por si só.

Ela é interpretada por uma **[routing policy / política de roteamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#política-de-roteamento)**.

Exemplo didático:

```text
confidence >= 0.90
→ primary_route

0.70 <= confidence < 0.90
→ escalation_route

confidence < 0.70
→ human_review
```

Esses limiares são apenas exemplos pedagógicos.

O princípio importante é:

```text
confidence
≠
ação automática

confidence
+ policy
→ rota
```


In [ ]:
def apply_routing_policy(decision: Decision):
    validate_decision(decision)

    if decision.confidence >= 0.90:
        route = "primary_route"
        reason = "high_confidence"
    elif decision.confidence >= 0.70:
        route = "escalation_route"
        reason = "medium_confidence"
    else:
        route = "human_review"
        reason = "low_confidence"

    return {
        "choice": decision.choice,
        "confidence": decision.confidence,
        "source": decision.source,
        "latency_ms": decision.latency_ms,
        "cost": decision.cost,
        "route_selected": route,
        "escalation_reason": reason,
    }


In [ ]:
decision_cases = [
    Decision("release", 0.96, "demo_provider", 18.0, 0.002),
    Decision("release", 0.82, "demo_provider", 18.0, 0.002),
    Decision("release", 0.61, "demo_provider", 18.0, 0.002),
]

routing_results = [apply_routing_policy(d) for d in decision_cases]
display(pd.DataFrame(routing_results))


### Interpretação

O mesmo `choice="release"` pode produzir rotas diferentes porque o workflow considera também a confiança.

Isso permite testar separadamente:

```text
DecisionProvider
→ qualidade da decisão

Routing Policy
→ regra de encaminhamento

Workflow
→ execução da rota
```

Essa separação é valiosa porque podemos substituir o produtor da decisão sem reescrever toda a lógica de execução.

Em um experimento AUTHOR / EVIDENCE, poderíamos medir também:

- calibração da confiança;
- Brier Score;
- ECE;
- coverage;
- escalation rate;
- custo;
- latência;
- utility.

Nesta aula, porém, esses valores permanecem didáticos.


## 7. [Retry](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#retry)

Retry é uma **nova tentativa controlada** após uma falha recuperável.

Ele precisa de:

```text
erro considerado recuperável
+ limite de tentativas
+ condição de parada
+ observabilidade
```

Retry não é sinônimo de `while True`.


In [ ]:
TRANSIENT_FAILURES = {"WF-RETRY": 1}

def build_release_note_unstable(state: WorkflowState):
    remaining = TRANSIENT_FAILURES.get(state.workflow_id, 0)
    if remaining > 0:
        TRANSIENT_FAILURES[state.workflow_id] = remaining - 1
        raise ConnectionError("temporary_dependency_failure")
    return build_release_note(state)

def run_with_retry(state, step_name, fn, max_attempts=2):
    for _ in range(max_attempts):
        ok = run_step(state, step_name, fn)
        if ok:
            return True
        last_error = state.errors[-1]["error"]
        if "ConnectionError" not in last_error:
            return False
    return False


In [ ]:
retry_state = WorkflowState(
    workflow_id="WF-RETRY",
    lesson_id="17",
    approved=True,
)

run_step(retry_state, "validate_release_request", validate_release_request)
run_step(retry_state, "get_lesson_status", get_lesson_status)

retry_ok = run_with_retry(
    retry_state,
    "build_release_note",
    build_release_note_unstable,
    max_attempts=2,
)

print("retry success:", retry_ok)
print("attempts:", retry_state.attempts["build_release_note"])
display(pd.DataFrame(retry_state.events))


## 8. [Idempotência](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#idempotência)

Uma operação idempotente pode ser repetida sem produzir um efeito adicional indesejado.

No nosso laboratório:

```text
workflow_id = chave de idempotência
```

Se a etapa de publicação for chamada novamente com o mesmo `workflow_id`, ela deve reconhecer que a publicação já ocorreu.


In [ ]:
idem = WorkflowState(
    workflow_id="WF-IDEM",
    lesson_id="17",
    approved=True,
)
idem.outputs["build_release_note"] = {"release_note": "Release controlada."}

first = publish_release_note(idem)
second = publish_release_note(idem)

print("primeira chamada:", first)
print("segunda chamada:", second)
print("publicações armazenadas:", len(PUBLISHED_NOTES))


## 9. [Approval gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#gate-de-aprovação)

Approval gate significa que o fluxo pode parar antes de uma etapa com efeito relevante.

Governar antes da execução quer dizer:

> submeter a ação proposta a regras de validação, autorização e política antes de permitir que ela produza efeitos reais.

Vamos executar o fluxo sem aprovação.


In [ ]:
pending_approval = WorkflowState(
    workflow_id="WF-APPROVAL",
    lesson_id="17",
    approved=None,
)

run_workflow(pending_approval)

print("status:", pending_approval.status)
print("current_step:", pending_approval.current_step)
print("último erro:", pending_approval.errors[-1])


## 10. Retomando o workflow

Depois de uma pausa, não precisamos necessariamente executar tudo novamente.

Podemos:

```text
inspecionar estado
→ resolver a condição
→ retomar da etapa adequada
```

No exemplo, concederemos aprovação e retomaremos a partir do approval gate.


In [ ]:
pending_approval.approved = True

approval_index = [name for name, _ in WORKFLOW_STEPS].index("approval_gate")
run_workflow(pending_approval, start_at=approval_index)

print("status final:", pending_approval.status)
print("current_step:", pending_approval.current_step)
display(pd.DataFrame(pending_approval.events))


## 11. Aprovação negada

Approval gate não existe apenas para pausar.

Ele também precisa permitir uma decisão explícita de **não executar**.


In [ ]:
rejected = WorkflowState(
    workflow_id="WF-REJECTED",
    lesson_id="17",
    approved=False,
)

run_workflow(rejected)

print("status:", rejected.status)
print("current_step:", rejected.current_step)
print("último erro:", rejected.errors[-1]["error"])


## 12. [Observabilidade de workflows](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#observabilidade-de-workflows)

Um workflow observável permite reconstruir:

```text
qual etapa executou?
quantas vezes?
quanto demorou?
qual erro ocorreu?
onde parou?
qual checkpoint existe?
```

Vamos consolidar o log da execução com retry.


In [ ]:
workflow_log = pd.DataFrame(retry_state.events)
display(workflow_log)


## 13. Failure taxonomy

Um workflow pode falhar de maneiras diferentes:

| Falha | Exemplo |
|---|---|
| step failure | uma função lança exceção |
| validation failure | entrada inválida |
| retry exhaustion | limite de tentativas atingido |
| duplicate execution risk | side effect repetido |
| state inconsistency | estado não representa o progresso real |
| approval rejection | ação não autorizada |
| downstream dependency failure | serviço dependente indisponível |

Dizer apenas “o workflow falhou” é tão insuficiente quanto dizer “a tool falhou”.


## 14. Laboratório — Recovery

Você recebeu este estado:

```text
workflow = WF-RECOVERY
lesson = 17
approval = required
status = paused
```

Sua tarefa:

1. identificar onde o fluxo parou;
2. verificar o último erro;
3. conceder aprovação;
4. retomar da etapa correta;
5. confirmar que a publicação ocorreu uma única vez.


In [ ]:
# Sua resposta aqui.
# Use WorkflowState, run_workflow e WORKFLOW_STEPS.


### Dica

Procure a etapa `approval_gate` em `WORKFLOW_STEPS` e use seu índice como `start_at`.


In [ ]:
recovery = WorkflowState(
    workflow_id="WF-RECOVERY",
    lesson_id="17",
    approved=None,
)

run_workflow(recovery)

recovery.approved = True
resume_index = [name for name, _ in WORKFLOW_STEPS].index("approval_gate")
run_workflow(recovery, start_at=resume_index)

publication = recovery.outputs["publish_release_note"]

print("status:", recovery.status)
print("publication:", publication)


## 15. Architecture Decision Lab — workflow ou agente?

Compare:

```text
funções manuais
vs
workflow determinístico
vs
agente autônomo
```

Para cada cenário, escolha a arquitetura mínima suficiente.

Critérios:

- sequência conhecida?
- regras estáveis?
- necessidade de adaptação dinâmica?
- risco?
- observabilidade?
- custo de falha?


In [ ]:
workflow_cases = pd.DataFrame([
    {"case": "A", "situation": "Validar pedido, calcular valor e gerar recibo sob regras fixas.", "sequence_known": True, "dynamic_reasoning": False},
    {"case": "B", "situation": "Escolher livremente entre dezenas de estratégias com base em contexto aberto.", "sequence_known": False, "dynamic_reasoning": True},
    {"case": "C", "situation": "Executar checklist de publicação com aprovação humana.", "sequence_known": True, "dynamic_reasoning": False},
    {"case": "D", "situation": "Planejar autonomamente uma investigação com ferramentas heterogêneas.", "sequence_known": False, "dynamic_reasoning": True},
])

display(workflow_cases)


### Sua tarefa

Escolha:

- `direct_functions`
- `deterministic_workflow`
- `agentic_system`

e justifique.

Lembre-se:

> **Maior autonomia não é automaticamente maior utility.**


In [ ]:
architecture_solution = {
    "A": "deterministic_workflow",
    "B": "agentic_system",
    "C": "deterministic_workflow",
    "D": "agentic_system",
}

workflow_cases["reference_choice"] = workflow_cases["case"].map(architecture_solution)
display(workflow_cases)


## 16. Síntese

Nesta aula, evoluímos de:

```text
tool isolada
→ decisão tipada
→ routing policy
→ sequência explícita
→ estado
→ retry
→ checkpoint
→ idempotência
→ approval
→ recovery
```

Um **workflow determinístico** coordena capacidades sob uma sequência explícita, estado observável e políticas previsíveis. Quando recebe decisões, elas devem entrar por contratos explícitos e ser interpretadas por uma routing policy antes da execução.

Ele deve ser preferido a maior autonomia quando já resolve o problema com menor risco e complexidade.


## 17. Ponte para MCP

Agora temos:

```text
Tool Registry
→ Deterministic Workflow
```

A próxima aula introduzirá:

```text
Model Context Protocol (MCP)
```

O objetivo será padronizar como capacidades são expostas e descobertas, sem abandonar os princípios já aprendidos:

- contratos;
- validação;
- governança;
- observabilidade;
- execução controlada.

MCP não substituirá essas ideias. Ele deverá organizá-las em uma camada de integração padronizada.


## 18. Reprodutibilidade

A aula foi desenhada para:

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
```

Todas as falhas e publicações são simuladas localmente.


---

## Continue no TIL

← **[Anterior: Aula 17 — Tool Use, Function Calling and Contracts](https://www.kaggle.com/code/pedrogentil/til-17-tool-use-function-calling-and-contracts)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **[Próxima: Aula 19 — Model Context Protocol (MCP) — em preparação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/ROADMAP.md)** →
